<a href="https://colab.research.google.com/github/roism126/DeepLearning/blob/main/Copy_of_Exercise_10_food_Mastery_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import tensorflow as tf
import matplotlib.pyplot as plt

from zipfile import ZipFile
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras import Model
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print("TensorFlow version:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.20.0
GPU tersedia: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from zipfile import ZipFile
import os

BASE_DIR = '/content/drive/MyDrive/Digitalent Data Science/food_dataset'

train_zip = os.path.join(
    BASE_DIR,
    '10_food_classes_10_percent.zip'
)

with ZipFile(train_zip, 'r') as zip_ref:
    zip_ref.extractall('/content/')

print("Isi direktori 10_food_classes_10_percent:")
print(os.listdir('10_food_classes_10_percent'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Isi direktori 10_food_classes_10_percent:
['train', 'test']


In [ ]:
training_set_dir = '10_food_classes_10_percent/train/'
test_set_dir = '10_food_classes_10_percent/test/'

print("Kategori pada Training Set:", sorted(os.listdir(training_set_dir)))

Kategori pada Training Set: ['chicken_curry', 'chicken_wings', 'fried_rice', 'grilled_salmon', 'hamburger', 'ice_cream', 'pizza', 'ramen', 'steak', 'sushi']


In [ ]:
# Training Set (70% dari data di folder train/)
training_set = image_dataset_from_directory(
    directory=training_set_dir,
    image_size=(224, 224),
    label_mode='categorical',
    batch_size=32,
    validation_split=0.3,
    subset='training',
    seed=42
)

# Validation Set (30% dari data di folder train/, split sama dengan training_set)
validation_set = image_dataset_from_directory(
    directory=training_set_dir,
    image_size=(224, 224),
    label_mode='categorical',
    batch_size=32,
    validation_split=0.3,
    subset='validation',
    seed=42
)

# Test Set (folder terpisah, tidak displit)
test_set = image_dataset_from_directory(
    directory=test_set_dir,
    image_size=(224, 224),
    label_mode='categorical'
)

class_names = training_set.class_names
print("Kelas:", class_names)

Found 750 files belonging to 10 classes.
Using 525 files for training.
Found 750 files belonging to 10 classes.
Using 225 files for validation.
Found 1552 files belonging to 8 classes.
Kelas: ['chicken_curry', 'chicken_wings', 'fried_rice', 'grilled_salmon', 'hamburger', 'ice_cream', 'pizza', 'ramen', 'steak', 'sushi']


In [ ]:
print(sorted(os.listdir(test_set_dir)))

['chicken_curry', 'fried_rice', 'grilled_salmon', 'ice_cream', 'pizza', 'ramen', 'steak', 'sushi']


In [ ]:
# --- Membangun Test Set independen & lengkap 10 kategori ---
# Hanya jalankan blok ini jika test_set_dir asli terbukti TIDAK lengkap 10 kategori.

import shutil
import random

random.seed(42)

# 1) DATASET FULL → SUMBER INDEPENDENT TEST - hanya untuk sumber gambar test set
full_zip = os.path.join(
    BASE_DIR,
    '10_food_classes.zip'
)

full_extract_path = '/content/10_food_classes_full'

if not os.path.exists(full_extract_path):
    with ZipFile(full_zip, 'r') as zip_ref:
        zip_ref.extractall(full_extract_path)

# with ZipFile('10_food_classes_full.zip') as zip_ref:
#         zip_ref.extractall('10_food_classes_full')

full_train_dir = '10_food_classes_full/10_food_classes/train'
print("Kategori pada dataset penuh:", sorted(os.listdir(full_train_dir)))

# 2) Untuk tiap kategori, ambil gambar yang BELUM dipakai di training_set (10_percent)
#    dengan membandingkan nama file -> mencegah data leakage antara train & test
custom_test_dir = 'test_set_custom'
N_PER_CLASS = 100  # jumlah gambar test per kategori, sesuaikan jika perlu

if os.path.exists(custom_test_dir):
    shutil.rmtree(custom_test_dir)
os.makedirs(custom_test_dir, exist_ok=True)

for category in sorted(os.listdir(full_train_dir)):
    used_files = set(os.listdir(os.path.join(training_set_dir, category)))
    all_full_files = os.listdir(os.path.join(full_train_dir, category))
    unused_files = [f for f in all_full_files if f not in used_files]

    if len(unused_files) < N_PER_CLASS:
        print(f"[Peringatan] {category}: hanya {len(unused_files)} gambar tersedia (< {N_PER_CLASS})")

    sampled = random.sample(unused_files, min(N_PER_CLASS, len(unused_files)))

    dest_dir = os.path.join(custom_test_dir, category)
    os.makedirs(dest_dir, exist_ok=True)
    for fname in sampled:
        shutil.copy(
            os.path.join(full_train_dir, category, fname),
            os.path.join(dest_dir, fname)
        )

print("\nTest set custom selesai dibangun.")
print("Kategori:", sorted(os.listdir(custom_test_dir)))
print("Jumlah gambar per kategori:", {c: len(os.listdir(os.path.join(custom_test_dir, c))) for c in os.listdir(custom_test_dir)})

Kategori pada dataset penuh: ['chicken_curry', 'chicken_wings', 'fried_rice', 'grilled_salmon', 'hamburger', 'ice_cream', 'pizza', 'ramen', 'steak', 'sushi']

Test set custom selesai dibangun.
Kategori: ['chicken_curry', 'chicken_wings', 'fried_rice', 'grilled_salmon', 'hamburger', 'ice_cream', 'pizza', 'ramen', 'steak', 'sushi']
Jumlah gambar per kategori: {'fried_rice': 100, 'pizza': 100, 'sushi': 100, 'chicken_curry': 100, 'steak': 100, 'ramen': 100, 'hamburger': 100, 'chicken_wings': 100, 'grilled_salmon': 100, 'ice_cream': 100}


In [ ]:
# 3) Timpa test_set dengan versi yang lengkap & independen
test_set = image_dataset_from_directory(
    directory='test_set_custom',
    image_size=(224, 224),
    label_mode='categorical'
)

print("Test set final -> kelas terdeteksi:", test_set.class_names)
assert len(test_set.class_names) == 10, "Masih belum 10 kategori, cek ulang sumber data!"

Found 1000 files belonging to 10 classes.
Test set final -> kelas terdeteksi: ['chicken_curry', 'chicken_wings', 'fried_rice', 'grilled_salmon', 'hamburger', 'ice_cream', 'pizza', 'ramen', 'steak', 'sushi']


In [ ]:
pretrained_model = EfficientNetV2B0(
    weights='imagenet',
    include_top=False,
    pooling='avg'
)

# Input layer sesuai ukuran gambar (224x224, 3 channel warna)
inputs = Input(shape=(224, 224, 3), name='input_layer')

# Callback yang dipakai di ketiga strategi
es_callback = EarlyStopping(monitor='val_loss', patience=5)

24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
# Strategi 3 — Full Freeze (Semua Layer Pre-trained Dibekukan)
# Kunci seluruh parameter pre-trained model
pretrained_model.trainable = False

# Sambungkan input -> pretrained model -> output baru (10 kelas)
x = pretrained_model(inputs)
outputs = Dense(units=10, activation='softmax', name='output_layer')(x)

model_3 = Model(inputs, outputs, name='model_3')

model_3.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model_3.summary()

Model: "model_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-b0 (Functional)  │ (None, 1280)           │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,932,122 (22.63 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 5,919,312 (22.58 MB)

In [ ]:
mpt_callback_3 = ModelCheckpoint(filepath='best_model_3.keras', save_best_only=True)

history_3 = model_3.fit(
    training_set,
    epochs=100,
    validation_data=validation_set,
    callbacks=[es_callback, mpt_callback_3]
)

Epoch 1/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 74s 2s/step - accuracy: 0.2838 - loss: 2.0503 - val_accuracy: 0.5378 - val_loss: 1.6727
Epoch 2/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 137ms/step - accuracy: 0.6876 - loss: 1.3701 - val_accuracy: 0.6978 - val_loss: 1.2524
Epoch 3/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 137ms/step - accuracy: 0.7905 - loss: 1.0208 - val_accuracy: 0.7556 - val_loss: 1.0286
Epoch 4/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 188ms/step - accuracy: 0.8381 - loss: 0.8082 - val_accuracy: 0.7733 - val_loss: 0.9104
Epoch 5/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 138ms/step - accuracy: 0.8552 - loss: 0.6928 - val_accuracy: 0.7911 - val_loss: 0.8266
Epoch 6/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 137ms/step - accuracy: 0.8857 - loss: 0.5967 - val_accuracy: 0.8000 - val_loss: 0.7736
Epoch 7/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 139ms/step - accuracy: 0.8762 - loss: 0.5388 - val_accuracy: 0.8267 - val_loss: 0.7336
Epoch 8/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 155ms/step - accuracy: 0.8933 - loss: 0.4948 - val_accura

In [ ]:
best_model_3 = load_model('best_model_3.keras')
result_3 = best_model_3.evaluate(test_set)
print(f"Strategi 3 -> Test Loss: {result_3[0]:.4f}, Test Accuracy: {result_3[1]:.4f}")

32/32 ━━━━━━━━━━━━━━━━━━━━ 22s 388ms/step - accuracy: 0.8510 - loss: 0.5038
Strategi 3 -> Test Loss: 0.5038, Test Accuracy: 0.8510


In [ ]:
# Strategi 2 — Buka 5 Layer Terakhir (Partial Fine-Tuning)

pretrained_model.trainable = True

# Bekukan semua layer kecuali 5 layer terakhir
for layer in pretrained_model.layers[:-5]:
    layer.trainable = False

x = pretrained_model(inputs)
outputs = Dense(units=10, activation='softmax', name='output_layer')(x)

model_2 = Model(inputs, outputs, name='model_2')

model_2.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model_2.summary()

Model: "model_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-b0 (Functional)  │ (None, 1280)           │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,932,122 (22.63 MB)

 Trainable params: 261,130 (1020.04 KB)

 Non-trainable params: 5,670,992 (21.63 MB)

In [ ]:
mpt_callback_2 = ModelCheckpoint(filepath='best_model_2.keras', save_best_only=True)

history_2 = model_2.fit(
    training_set,
    epochs=100,
    validation_data=validation_set,
    callbacks=[es_callback, mpt_callback_2]
)

Epoch 1/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 64s 2s/step - accuracy: 0.5410 - loss: 1.5948 - val_accuracy: 0.7822 - val_loss: 0.9323
Epoch 2/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 139ms/step - accuracy: 0.8438 - loss: 0.6694 - val_accuracy: 0.8222 - val_loss: 0.6647
Epoch 3/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 227ms/step - accuracy: 0.9048 - loss: 0.4295 - val_accuracy: 0.8267 - val_loss: 0.6006
Epoch 4/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 143ms/step - accuracy: 0.9371 - loss: 0.3043 - val_accuracy: 0.8578 - val_loss: 0.5641
Epoch 5/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 150ms/step - accuracy: 0.9505 - loss: 0.2364 - val_accuracy: 0.8400 - val_loss: 0.5490
Epoch 6/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 196ms/step - accuracy: 0.9771 - loss: 0.1777 - val_accuracy: 0.8444 - val_loss: 0.5420
Epoch 7/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 7s 307ms/step - accuracy: 0.9848 - loss: 0.1395 - val_accuracy: 0.8356 - val_loss: 0.5362
Epoch 8/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 154ms/step - accuracy: 0.9886 - loss: 0.1114 - val_accura

In [ ]:
best_model_2 = load_model('best_model_2.keras')
result_2 = best_model_2.evaluate(test_set)
print(f"Strategi 2 -> Test Loss: {result_2[0]:.4f}, Test Accuracy: {result_2[1]:.4f}")

32/32 ━━━━━━━━━━━━━━━━━━━━ 16s 177ms/step - accuracy: 0.8630 - loss: 0.4636
Strategi 2 -> Test Loss: 0.4636, Test Accuracy: 0.8630


In [ ]:
# Strategi 1 — Full Fine-Tuning (Semua Layer Dibuka)

pretrained_model.trainable = True  # semua layer dibuka, tanpa pengecualian

# PENTING: layer.trainable individual yang di-set False di Strategi 2
# TIDAK otomatis kembali True hanya dengan baris di atas.
# Harus di-reset eksplisit per layer, jika tidak, Strategi 1 diam-diam
# hanya mengulang Strategi 2 (hanya 5 layer terakhir yang benar-benar trainable).
for layer in pretrained_model.layers:
    layer.trainable = True

x = pretrained_model(inputs)
outputs = Dense(units=10, activation='softmax', name='output_layer')(x)

model_1 = Model(inputs, outputs, name='model_1')

model_1.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model_1.summary()

Model: "model_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-b0 (Functional)  │ (None, 1280)           │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,932,122 (22.63 MB)

 Trainable params: 5,871,514 (22.40 MB)

 Non-trainable params: 60,608 (236.75 KB)

In [ ]:
mpt_callback_1 = ModelCheckpoint(filepath='best_model_1.keras', save_best_only=True)

history_1 = model_1.fit(
    training_set,
    epochs=100,
    validation_data=validation_set,
    callbacks=[es_callback, mpt_callback_1]
)


Epoch 1/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 163s 4s/step - accuracy: 0.5352 - loss: 1.5056 - val_accuracy: 0.6978 - val_loss: 0.9121
Epoch 2/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 29s 308ms/step - accuracy: 0.9048 - loss: 0.3514 - val_accuracy: 0.7022 - val_loss: 0.8419
Epoch 3/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 158ms/step - accuracy: 0.9752 - loss: 0.1027 - val_accuracy: 0.7111 - val_loss: 0.9309
Epoch 4/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 156ms/step - accuracy: 0.9886 - loss: 0.0471 - val_accuracy: 0.6933 - val_loss: 1.0460
Epoch 5/100
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 183ms/step - accuracy: 0.9886 - loss: 0.0433 - val_accuracy: 0.7644 - val_loss: 0.8758


In [ ]:
best_model_1 = load_model('best_model_1.keras')
result_1 = best_model_1.evaluate(test_set)
print(f"Strategi 1 -> Test Loss: {result_1[0]:.4f}, Test Accuracy: {result_1[1]:.4f}")

32/32 ━━━━━━━━━━━━━━━━━━━━ 16s 209ms/step - accuracy: 0.7480 - loss: 0.7727
Strategi 1 -> Test Loss: 0.7727, Test Accuracy: 0.7480


In [ ]:
import pandas as pd

def count_trainable_params(model):
    return sum([tf.size(w).numpy() for w in model.trainable_weights])

comparison = pd.DataFrame({
    'Strategi': ['Strategi 3 (Full Freeze)', 'Strategi 2 (5 Layer Terakhir)', 'Strategi 1 (Full Fine-tune)'],
    'Trainable Params': [
        count_trainable_params(best_model_3),
        count_trainable_params(best_model_2),
        count_trainable_params(best_model_1)
    ],
    'Test Loss': [result_3[0], result_2[0], result_1[0]],
    'Test Accuracy': [result_3[1], result_2[1], result_1[1]]
})

comparison

,Strategi,Trainable Params,Test Loss,Test Accuracy
0,Strategi 3 (Full Freeze),12810,0.503803,0.851
1,Strategi 2 (5 Layer Terakhir),261130,0.463642,0.863
2,Strategi 1 (Full Fine-tune),5871514,0.772695,0.748


In [ ]:
plt.figure(figsize=(12, 4))
plt.bar(comparison['Strategi'], comparison['Test Accuracy'], color=['#4C72B0', '#55A868', '#C44E52'])
plt.ylabel('Test Accuracy')
plt.title('Perbandingan Akurasi Test - 3 Strategi Transfer Learning')
plt.xticks(rotation=45)
plt.ylim([0, 1])
plt.tight_layout()
plt.show()


# Kesimpulan

---


Strategi 3 (full freeze) sudah memberi hasil baik dengan biaya komputasi paling murah — cocok jadi baseline pertama untuk data terbatas.

Strategi 2 (buka beberapa layer terakhir) umumnya memberi peningkatan akurasi tanpa risiko overfitting besar — sweet spot untuk kebanyakan kasus dataset kecil-menengah.

Strategi 1 (full fine-tuning) berisiko overfitting jika data training tidak cukup besar untuk menyesuaikan jutaan parameter — perhatikan grafik training vs validation loss untuk mendeteksi gejala ini.

### Eksperimen lanjutan yang bisa dicoba:

---



Ganti pre-trained model: ResNet50, MobileNetV2, dll — bandingkan akurasi dan waktu training.

Coba dataset 10_food_classes_1_percent.zip untuk melihat efek data training yang lebih ekstrem sedikit.

Tambahkan layer augmentasi gambar (RandomFlip, RandomRotation) sebelum masuk ke pre-trained model untuk mengurangi overfitting pada Strategi 1.